# Batch #1 - xattn seed sweep (interactive run)

Eleven runs, end to end, on one GPU session:

| arm | runs | what it is |
|---|---|---|
| `A160` | seeds 42-46 | recorded-best budget (comment 160 / video 256), 5 model seeds = controls + noise floor |
| `AUX` | seeds 42-44 | `A160` + train-time `hate_speech` aux head (weight 0.5) = experimental arm |
| `B512` | seeds 42-44 | bigger token budget (512 / 512) = budget arm |

Every run uses `--split-seed 42`, so the validation split is identical across
runs and the saved probability dumps are directly ensemblable. Code is pinned
at `37c4c26` (the revision whose aux head was verified bit-identical with the
flag off).

**Checklist before Run All** (right sidebar -> Settings):

1. **Accelerator: GPU T4 x2**
2. **Internet: On** (git clone + pip + model download)
3. **Add Input -> Datasets -> `amnpham/lgbtqia`**

**How long:** ~2.5-3 h for the whole batch. A cell that is executing cannot
idle-timeout - keep the session running and don't press Stop.

**Resume:** each run is skipped when its history CSV already exists, so
re-running the batch cell continues where it stopped (within the same session).
A fresh session starts from scratch.

**When it prints `BATCH COMPLETE`:** download `Try/weights/results_small.zip`
from the **Output** panel (right side -> `/kaggle/working` -> `Try` ->
`weights`). That small zip contains every history CSV, probability dump and
summary - the big `*.pt` checkpoints are not needed.


In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/AmnO-O/Try.git'
PIN_COMMIT = '37c4c2645aea0dff1e749dc6f762f33d5e843d35'  # aux head, flag-off bit-identical


def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)


# Re-run safe: after the first run the cwd is already <working>/Try.
if os.path.basename(os.path.abspath('.')) != 'Try':
    if not os.path.isdir('Try'):
        sh(f'git clone {REPO_URL}')
    os.chdir('Try')
if PIN_COMMIT:
    sh('git fetch origin main')
    sh(f'git checkout {PIN_COMMIT}')
sh('git log -1 --format="%H %cd %s" --date=iso')
sh('pip install -q -r requirements.txt')
print('cwd:', os.getcwd(), flush=True)


## 1. Data + GPU check

Finds the StereoQueerEval TSVs under `/kaggle/input` (attach `amnpham/lgbtqia`),
points `LGBT_DATA_DIR` / `LGBT_WEIGHTS_DIR` at the right places and asserts a
GPU is present.


In [ ]:
import glob
import sys
import time

hits = sorted(glob.glob('/kaggle/input/**/StereoQueerEval_*_training.tsv', recursive=True))
assert hits, 'StereoQueerEval TSVs not found - attach the dataset (Add Input > amnpham/lgbtqia)'
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs('weights', exist_ok=True)
print('data dir:', os.environ['LGBT_DATA_DIR'], flush=True)
print('tsvs:', [os.path.basename(h) for h in hits], flush=True)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU T4 x2, then Run All'
print('gpu:', torch.cuda.get_device_name(0), flush=True)

sys.path.insert(0, os.path.abspath('src'))
import numpy as np
import pandas as pd
import data as data_mod
import model_xattn
from model import load_tokenizer
import train_xattn as tx


## 2. Run list and helpers

Same flags as the recorded best run (shared encoder, 3 decoder blocks, top-3
encoder blocks unfrozen, dropout 0.3, wd 0.01, head-lr 1e-3, backbone-lr 2e-5,
xattn-lr 1e-4). Only the model seed varies - plus the aux-head weight for `AUX`.
Split seed is 42 everywhere.


In [ ]:
BASE = dict(dropout=0.3, weight_decay=0.01, head_lr=1e-3,
            unfreeze_layers=3, layers=3, shared_encoder=True, pooled_v=False)

RUNS = []
A = dict(BASE, max_len=160, video_max_len=256)          # the recorded best budget
B = dict(BASE, max_len=512, video_max_len=512)          # budget arm
for s in (42, 43, 44, 45, 46):
    RUNS.append(dict(A, tag=f'A160_s{s}', seed=s))
for s in (42, 43, 44):
    RUNS.append(dict(A, tag=f'AUX_s{s}', seed=s, hate_aux_weight=0.5))
for s in (42, 43, 44):
    RUNS.append(dict(B, tag=f'B512_s{s}', seed=s))

# Validation set is the same for every run (split-seed 42), so the probability
# dumps are directly ensemblable.
dev = tx.resolve_device('cuda')
tok = load_tokenizer(model_xattn.DEFAULT_BACKBONE)
df_all = data_mod.load_frame(languages=['EN', 'IT', 'NL'])
_, df_val = data_mod.split_by_video(df_all, 0.1, 42)
print('val rows:', len(df_val), '| by lang:', df_val['lang'].value_counts().to_dict(), flush=True)
np.save('weights/val_langs.npy', df_val['lang'].to_numpy(dtype=str))


def train_one(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    if os.path.exists(hist):
        print(f"-- skip (history exists): {hist}", flush=True)
        return True
    aux = r.get('hate_aux_weight')
    aux_flag = f"--hate-aux-weight {aux} " if aux else ''
    cmd = (
        'python src/train_xattn.py --languages EN IT NL --epochs 15 --patience 5 --batch-size 32 '
        f"--max-len {r['max_len']} --video-max-len {r['video_max_len']} "
        f"--dropout {r['dropout']} --weight-decay {r['weight_decay']} --head-lr {r['head_lr']} "
        f"--unfreeze-layers {r['unfreeze_layers']} --layers {r['layers']} --no-pooled-v "
        f"{'--shared-encoder ' if r['shared_encoder'] else ''}"
        f"--split-seed 42 --seed {r['seed']} {aux_flag}"
        f"--out {out} --history {hist}"
    )
    print(f'$ {cmd}', flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f'== {r["tag"]}: exit {rc} after {(time.time() - t0) / 60:.1f} min', flush=True)
    if rc != 0 or not os.path.exists(hist):
        print(f'!! {r["tag"]} failed', flush=True)
        return False
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    print(f"   {r['tag']}: best epoch {int(h.loc[bi, 'epoch'])} | "
          f"val_macro_f1 {h.loc[bi, 'val_macro_f1']:.4f} | acc {h.loc[bi, 'val_acc']:.4f}", flush=True)
    return True


def dump_probs(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    if not os.path.exists(hist) or not os.path.exists(out):
        print(f'-- no probs for {r["tag"]}', flush=True)
        return
    net = model_xattn.build_model(
        model_xattn.DEFAULT_BACKBONE, unfreeze_layers=r['unfreeze_layers'],
        heads=4, layers=r['layers'], dropout=r['dropout'], ffn_mult=4,
        use_pooled_v=r['pooled_v'], separate_encoders=not r['shared_encoder'],
        keep_frozen_dropout=False, pool_init_scale=0.02,
        hate_aux=r.get('hate_aux_weight', 0) > 0,
    ).to(dev)
    net.load_state_dict(torch.load(out, map_location=dev))
    net.eval()
    loader = tx.make_loader(df_val, tok, 32, r['max_len'], r['video_max_len'])
    o = tx.collect(net, loader, dev, use_amp=True)
    np.save(f"weights/{r['tag']}_val_probs.npy", o['probs'])
    if r['tag'] == 'A160_s42':
        np.save('weights/val_labels.npy', o['labels'])
    m = tx.metrics_from(o['preds'], o['labels'])
    print(f"   {r['tag']}: reload-eval macro_f1 {m['macro_f1']:.4f} acc {m['acc']:.4f}", flush=True)
    del net
    torch.cuda.empty_cache()


def zip_small():
    """Bundle the small outputs (no .pt checkpoints) as weights/results_small.zip."""
    import zipfile
    small = [f for f in sorted(os.listdir('weights'))
             if f.endswith(('.csv', '.npy', '.json'))]
    with zipfile.ZipFile('weights/results_small.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in small:
            zf.write(os.path.join('weights', f), f)
    print(f'wrote weights/results_small.zip ({len(small)} files)', flush=True)


print('planned runs:', [r['tag'] for r in RUNS], flush=True)


## 3. Run the batch (~2.5-3 h)

Order: controls -> aux arm -> budget arm. A failed run does not abort the
batch, and an existing history CSV skips that run - so this cell can be
re-run after an interruption within the same session.


In [ ]:
t_start = time.time()
for i, r in enumerate(RUNS, 1):
    print(f'=== [{i}/{len(RUNS)}] {r["tag"]} ===', flush=True)
    try:
        train_one(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! train_one({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        dump_probs(r)
    except Exception as e:                                    # noqa: BLE001
        print(f'!! dump_probs({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
print(f'batch finished in {(time.time() - t_start) / 3600:.2f} h', flush=True)
zip_small()  # partial-results safety: the summary below can fail on a partial batch


## 4. Summary and seed ensembles

Per-run best-epoch table, then mean-probability ensembles (the primary lever).
Saved to `weights/run_summary.csv` and `weights/ensemble_summary.csv`, both
inside `results_small.zip`.


In [ ]:
try:
    rows = []
    for r in RUNS:
        hist = f"weights/{r['tag']}_history.csv"
        if os.path.exists(hist):
            h = pd.read_csv(hist)
            bi = h['val_macro_f1'].idxmax()
            rows.append(dict(tag=r['tag'], budget=f"{r['max_len']}/{r['video_max_len']}", seed=r['seed'],
                             epochs=len(h), best_epoch=int(h.loc[bi, 'epoch']),
                             best_f1=round(float(h.loc[bi, 'val_macro_f1']), 4),
                             best_acc=round(float(h.loc[bi, 'val_acc']), 4)))
        else:
            rows.append(dict(tag=r['tag'], budget=f"{r['max_len']}/{r['video_max_len']}", seed=r['seed'],
                             epochs=0, best_epoch=None, best_f1=None, best_acc=None))
    summary = pd.DataFrame(rows).sort_values(['budget', 'seed'])
    print(summary.to_string(index=False), flush=True)
    summary.to_csv('weights/run_summary.csv', index=False)

    labels = np.load('weights/val_labels.npy')

    def ens(tags):
        ps = [np.load(f"weights/{t}_val_probs.npy") for t in tags
              if os.path.exists(f"weights/{t}_val_probs.npy")]
        if len(ps) < 2:
            return None
        probs = np.mean(ps, axis=0)
        return tx.metrics_from(probs.argmax(1), labels), probs

    GROUPS = {
        'A160_x5': [f'A160_s{s}' for s in (42, 43, 44, 45, 46)],
        'A160_x3': [f'A160_s{s}' for s in (42, 43, 44)],
        'AUX_x3': [f'AUX_s{s}' for s in (42, 43, 44)],
        'B512_x3': [f'B512_s{s}' for s in (42, 43, 44)],
        'A160_x3+AUX_x3': [f'A160_s{s}' for s in (42, 43, 44)] + [f'AUX_s{s}' for s in (42, 43, 44)],
        'A160_x5+AUX_x3': [f'A160_s{s}' for s in (42, 43, 44, 45, 46)] + [f'AUX_s{s}' for s in (42, 43, 44)],
        'A160_x3+B512_x3': [f'A160_s{s}' for s in (42, 43, 44)] + [f'B512_s{s}' for s in (42, 43, 44)],
    }
    ens_rows = []
    for name, tags in GROUPS.items():
        res = ens(tags)
        if res is None:
            continue
        m, probs = res
        ens_rows.append(dict(ensemble=name, n_members=sum(
            os.path.exists(f"weights/{t}_val_probs.npy") for t in tags),
            macro_f1=round(float(m['macro_f1']), 4), acc=round(float(m['acc']), 4)))
        print(f"ENSEMBLE {name}: macro_f1 {m['macro_f1']:.4f} | acc {m['acc']:.4f}", flush=True)
    pd.DataFrame(ens_rows).to_csv('weights/ensemble_summary.csv', index=False)

    # per-language of the full mixed ensemble
    res = ens(GROUPS['A160_x3+B512_x3'])
    if res is not None:
        _, probs = res
        langs = np.load('weights/val_langs.npy')
        for lang in sorted(set(langs)):
            sel = langs == lang
            m = tx.metrics_from(probs.argmax(1)[sel], labels[sel])
            print(f"  [mixed ensemble {lang}] macro_f1 {m['macro_f1']:.4f} | acc {m['acc']:.4f}", flush=True)
except Exception as e:                                        # noqa: BLE001
    print(f'!! summary failed: {type(e).__name__}: {e}', flush=True)

print('files in weights/:', sorted(os.listdir('weights')), flush=True)
zip_small()  # refreshed copy including run_summary.csv / ensemble_summary.csv
print('BATCH COMPLETE - download Try/weights/results_small.zip from the Output panel', flush=True)


## Done

Download `results_small.zip` (Output panel, `Try/weights/`) and hand it over -
it has everything needed for the analysis: per-run histories, validation
probability dumps (the basis for the paired cluster bootstrap) and both
summary CSVs. The session can then be stopped.
